# Term Frequency – Inverse Document Frequency (TF–IDF)
## 1. Introduction

Consider the following academic abstracts:

**Paper A (Machine Learning)**
> "This paper presents a novel neural network architecture for image classification. The network uses convolutional layers…"

**Paper B (Cryptography)**
> "This paper presents a new encryption algorithm for secure communication. The algorithm uses prime factorization…"

How could a model tell these apart?

A naive approach is to simply count how many times each word appears and compare those counts:

|             | this | paper | presents | neural | network | encryption | algorithm |
|:------------|:----:|:-----:|:--------:|:------:|:-------:|:----------:|:---------:|
| **Paper A** |  1   |   1   |    1     |   2    |    2    |     0      |     0     |
| **Paper B** |  1   |   1   |    1     |   0    |    0    |     2      |     2     |

The issue is clear: words like *"this," "paper,"* and *"presents"* appear everywhere and carry no distinctive meaning.
The words *"neural," "network," "encryption,"* and *"algorithm"* actually define the topics, yet all words are treated equally in a raw count.

TF–IDF (Term Frequency–Inverse Document Frequency) solves this by rewarding words that are **frequent within a document** but **rare across the corpus**.
It measures two complementary properties of words:

---

### 1.1 Term Frequency (TF)

$$
\mathrm{TF}(w, d) \;=\; \frac{\text{count}(w,d)}{\sum_{w'} \text{count}(w',d)}
$$

**Example:**
If Paper A has 100 words and *"neural"* appears twice, then
$$\mathrm{TF}(\text{"neural"}, \text{Paper A}) = \tfrac{2}{100} = 0.02.$$

---

### 1.2 Inverse Document Frequency (IDF)

$$
\mathrm{IDF}(w) \;=\; \log\!\left(\frac{N}{n_w}\right)
$$

Where \(N\) is the total number of documents and \(n_w\) is the number of documents containing the word \(w\).

**Example:** Using a corpus of \(N=1000\) papers:

| Word          | Docs Containing |        IDF         |
|:--------------|:---------------:|:------------------:|
| the           |      1000       | log(1000/1000) = 0 |
| convolutional |       50        |  log(1000/50) = 3  |

Common words receive low IDF ≈ 0; rare words receive high IDF values.
*(Many libraries use smoothing, e.g., \(\log\!\big(\frac{N+1}{n_w+1}\big)+1\), to avoid edge cases.)*

---

### 1.3 Combining Them

$$
\mathrm{TF\text{-}IDF}(w, d) \;=\; \mathrm{TF}(w,d)\times \mathrm{IDF}(w)
$$

This produces a weighted vector where distinctive terms get higher scores and common filler terms fade toward zero.
The resulting vectors are more **separable**, allowing classifiers to distinguish topics like machine learning versus cryptography using only word statistics.

---
### 1.4 Important Topics

#### Stop Words
Stop words are widespread words used to glue sentences together but tell us little about the meaning of the text. Such words include `the`, `is`, `a`, `for`, etc., and appear in all forms of literature including fiction, research, blogging, etc. These values will score very low and therefore influence very little to the final classification. In training stop words were set to english to remove these words from the text when creating the bag of words, enabling the model to only consider more influential words.

#### Sparsity
Sparsity refers to the proportion of zero values in a matrix. In TF-IDF matrices, sparsity is extremely high because:

1. **Large vocabulary**: With max_features=100,000, each document becomes a vector with 100 thousand dimensions
2. **Limited document vocabulary**: A typical academic paper uses only 200-500 unique terms
3. **Zero dominance**: 99%+ of the vocabulary doesn't appear in any given document

**Mathematical example**: If a document contains 300 unique words out of 100 thousand-term vocabulary:
- Sparsity = (100,000–300) / 100 thousand = **99.7% sparse**

This extreme sparsity provides crucial computational advantages:
- **Memory efficiency**: Sparse matrices store only non-zero values, reducing memory from ~20GB to ~200MB
- **Computational speed**: Operations skip zero multiplications, dramatically reducing computation time
- **Scalability**: Enables processing of large vocabularies that would be impossible with dense representations

This foundation will let us explore how TF–IDF transforms raw text into meaningful numeric representations for downstream models such as logistic regression, SVMs, or neural classifiers.

---

### 1.5 Imports

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import os
import json
from datetime import datetime

# Create the results directory
results_dir = r'/data/results/tfidf'
os.makedirs(results_dir, exist_ok=True)
print(f"Results will be saved to: {results_dir}")


## 2. Load & Split Data

In [ ]:
print("Splitting data: using val in training data for final training")
data = pd.read_parquet('../data/processed/arxiv_text.parquet')

X_train = data[data['split'] == 'train']
y_train = data[data['split'] == 'train'].label

X_val = data[data['split'] == 'valid']
y_val = data[data['split'] == 'valid'].label

X_test = data[data['split'] == 'test']
y_test = data[data['split'] == 'test'].label

print(f"The shape of the training data is X train: {X_train.shape} and y train: {y_train.shape}")
print(f"The shape of the training data is X val: {X_val.shape} and y val: {y_val.shape}")
print(f"The shape of the training data is X test: {X_test.shape} and y test: {y_test.shape}")


## 3. TF-IDF Grid Search: Optimizing N-gram Range and Vocabulary Size
### 3.1 Simple Grid Search

In [ ]:
print("Start training")
gs_results = []
experiment_count = 0
total_experiments = 9

for (x, y) in [(1,2), (1,3), (1,5)]:
    for max_features in [1000, 10000, 100000]:
        experiment_count += 1
        print(f"Experiment {experiment_count}/{total_experiments}: ngram_range=({x},{y}), max_features={max_features}")
        
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(x, y),
            stop_words='english'
        )

        model = LogisticRegression(max_iter=10000)

        X_train_ngram = vectorizer.fit_transform(X_train['text'])
        X_val_ngram = vectorizer.transform(X_val['text'])

        # Calculate sparsity for analysis
        sparsity = 1.0 - (X_train_ngram.nnz / (X_train_ngram.shape[0] * X_train_ngram.shape[1]))
        print(f"  Matrix shape: {X_train_ngram.shape}, Sparsity: {sparsity:.3f}")

        model.fit(X_train_ngram, y_train)

        y_pred = model.predict(X_val_ngram)
        gs_results.append({
            'ngram_range': vectorizer.ngram_range,
            'max_features': vectorizer.max_features,
            'accuracy': accuracy_score(y_val, y_pred),
            'macro_f1': f1_score(y_val, y_pred, average='macro'),
            'sparsity': sparsity,
            'vocabulary_size': len(vectorizer.vocabulary_)
        })
        
        print(f"  Accuracy: {gs_results[-1]['accuracy']:.4f}, F1-macro: {gs_results[-1]['macro_f1']:.4f}")


In [ ]:
df_results = pd.DataFrame(gs_results)
best_params = df_results.sort_values(by="accuracy", ascending=False).iloc[0]
print(f"Best parameters by accuracy: {best_params}")

# Show all results sorted by F1-macro for analysis
print("\nAll results sorted by F1-macro:")
print("="*70)
sorted_results = df_results.sort_values("macro_f1", ascending=False)
for idx, row in sorted_results.iterrows():
    print(f"ngram_range={row['ngram_range']}, max_features={row['max_features']:6d}: "
          f"F1={row['macro_f1']:.4f}, Acc={row['accuracy']:.4f}, Sparsity={row['sparsity']:.3f}")


### 3.2 Training & Evaluation Using Best Parameters

In [ ]:
print("Splitting data: using val in training data for final training")
X_train = data[data['split'] != 'test']
y_train = data[data['split'] != 'test'].label

X_test = data[data['split'] == 'test']
y_test = data[data['split'] == 'test'].label

print(f"The shape of the training data is X train: {X_train.shape} and y train: {y_train.shape}")
print(f"The shape of the training data is X test: {X_test.shape} and y test: {y_test.shape}")


In [ ]:
import ast

print(f"Start training: using {best_params}")

# Ensure ngram_range is a proper tuple (fix the potential string issue)
if isinstance(best_params.ngram_range, str):
    ngram_range = ast.literal_eval(best_params.ngram_range)
else:
    ngram_range = best_params.ngram_range

vectorizer = TfidfVectorizer(
    max_features=best_params.max_features,
    ngram_range=ngram_range,
    stop_words='english'
)

model = LogisticRegression(max_iter=10000)

X_train_ngram = vectorizer.fit_transform(X_train['text'])
X_test_ngram = vectorizer.transform(X_test['text'])

# Calculate final sparsity metrics
final_sparsity = 1.0 - (X_train_ngram.nnz / (X_train_ngram.shape[0] * X_train_ngram.shape[1]))
print(f"Final training matrix shape: {X_train_ngram.shape}")
print(f"Final sparsity: {final_sparsity:.3f}")
print(f"Non-zero elements: {X_train_ngram.nnz:,} out of {X_train_ngram.shape[0] * X_train_ngram.shape[1]:,}")

model.fit(X_train_ngram, y_train)
y_pred = model.predict(X_test_ngram)

In [ ]:
results = pd.DataFrame({
    'model': 'LogisticRegression',
    'ngram_range': str(best_params.ngram_range),
    'max_iter': 10000,
    'max_features': best_params.max_features,
    'accuracy': accuracy_score(y_test, y_pred),
    'macro_f1': f1_score(y_test, y_pred, average='macro'),
    'sparsity': final_sparsity
}, index=[0])

tfidf_accuracy = results.iloc[0]['accuracy']
tfidf_f1 = results.iloc[0]['macro_f1']

print(f"Final Results")
print(results)

## 4. Analysis: What Makes the Difference

### 4.1 Hyperparameter Impact Analysis

In [ ]:
print("HYPERPARAMETER IMPACT ANALYSIS")
print("="*50)

# Analyze n-gram range impact
print("\n1. N-gram Range Analysis:")
for ngram in [(1,2), (1,3), (1,5)]:
    subset = df_results[df_results['ngram_range'] == ngram]
    avg_f1 = subset['macro_f1'].mean()
    max_f1 = subset['macro_f1'].max()
    print(f"  {ngram}: Average F1={avg_f1:.4f}, Max F1={max_f1:.4f}")

# Analyze vocabulary size impact
print("\n2. Vocabulary Size Analysis:")
for vocab_size in [1000, 10000, 100000]:
    subset = df_results[df_results['max_features'] == vocab_size]
    avg_f1 = subset['macro_f1'].mean()
    max_f1 = subset['macro_f1'].max()
    avg_sparsity = subset['sparsity'].mean()
    print(f"  {vocab_size:6d} features: Average F1={avg_f1:.4f}, Max F1={max_f1:.4f}, Avg Sparsity={avg_sparsity:.3f}")

# Best combination analysis
best_combo = df_results.loc[df_results['accuracy'].idxmax()]
print(f"\n3. Optimal Configuration:")
print(f"  N-gram range: {best_combo['ngram_range']}")
print(f"  Max features: {best_combo['max_features']}")
print(f"  Max features: {best_combo['max_features']}")
print(f"  Accuracy: {best_combo['accuracy']:.4f}")
print(f"  Test F1-macro: {best_combo['macro_f1']:.4f}")
print(f"  Sparsity: {best_combo['sparsity']:.3f}")


### 4.2 Key Success Factors

1. **Feature Engineering Excellence:**
   - TF-IDF weighting emphasizes discriminative terms
   - N-gram range captures both words and phrases
   - Stop word removal focuses on content-bearing terms

2. **Optimal Complexity Balance:**
   - Feature space large enough for coverage, but not too large for noise
   - Sparse representation enables efficient computation
   - Vocabulary size reflects unique, meaningful terms

3. **Text Representation Advantages:**
   - TF-IDF outperforms raw word counts by emphasizing rare, discriminative terms
   - Document normalization handles variable text lengths
   - Domain-specific terms receive higher semantic weighting

4. **Computational Efficiency:**
   - Sparse matrix storage provides significant memory savings
   - Linear models and sparse features enable fast training
   - Architecture scales efficiently with large vocabularies


### 4.3 Limitations and Future Directions

**Current Limitations:**
1. **Class Imbalance:** Still struggles with rare classes, reflected by the gap between macro and weighted F1 scores.
2. **Semantic Understanding:** Bag-of-words ignores word order and contextual meaning.
3. **Graph Structure:** Lacks citation network and relational information.
4. **Domain Overlap:** Academic papers often share similar vocabulary across fields, reducing discriminative power.

**Future Improvements:**
1. **Advanced Embeddings:** Integrate transformer-based models such as BERT or SciBERT to enhance semantic understanding.
2. **Graph Integration:** Incorporate citation networks and author relationships for richer structural context.
3. **Hierarchical Classification:** Utilize subject area taxonomies for multi-level categorization.
4. **Class Balancing:** Employ advanced sampling or weighting strategies to handle rare classes effectively.


In [ ]:
final_analysis = {
    'tfidf_performance': {
        'accuracy': float(tfidf_accuracy),
        'f1_macro': float(tfidf_f1),
    },
    'key_factors': {
        'optimal_ngram_range': best_combo['ngram_range'],
        'optimal_vocab_size': int(best_combo['max_features']),
        'sparsity_advantage': float(best_combo['sparsity']),
        'feature_engineering': 'TF-IDF weighting + stop word removal'
    }
}

# Save final analysis
analysis_path = os.path.join(results_dir, f'tfidf_analysis.json')
with open(analysis_path, 'w') as f:
    json.dump(final_analysis, f, indent=2)

print(f"\nAnalysis complete! Results saved to: {results_dir}")